# Exploring the repo-local Brightway store

Interactive inspection of the databases the workflow built under `build/lci/brightway`
(`constants.BRIGHTWAY_STORE`). Read-only: nothing here writes to the store and nothing
here feeds the pipeline — it is the console for answering "what is this activity, and
what is in it" when writing `data/lci_info.csv`.

Run it with the **`metal-lci`** kernel. If that kernel is not listed, register it once:

```bash
conda run -n metal-lci python -m ipykernel install --user --name metal-lci
```

## 1. Point Brightway at the repo-local store

`bw2data` resolves its data directory from `BRIGHTWAY2_DIR` **at import time**, so the
environment variable has to be set before the first `import bw2data` in this kernel.
If you have already imported it, restart the kernel.

In [2]:
import os
import sys
from pathlib import Path

import yaml

# Locate the repo root (the directory holding the snakefile) from wherever this runs.
root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "snakefile").exists()
)
sys.path.append(str(root / "scripts"))

from constants import BRIGHTWAY_STORE, RESOURCE_CATEGORY
from info_schemas import LciInfo

os.environ["BRIGHTWAY2_DIR"] = str(root / BRIGHTWAY_STORE)

config = yaml.safe_load((root / "config/default.yaml").read_text())
project = config["lci"]["project"]

import bw2data as bd

bd.projects.set_current(project)
print(bd.projects.dir)
print(list(bd.databases))

['biosphere', 'ecoinvent-3.10-cutoff', 'ei_cutoff_3.10_tiam-ucl_SSP2-RCP19_2040-corrected', 'ecoinvent-3.10-cutoff-aluminium-conductor', 'ei_cutoff_3.10_tiam-ucl_SSP2-RCP19_2040']


The two databases the workflow builds are named from the config: the imported
ecoinvent, and the premise prospective database derived from it.

In [3]:
ei = config["lci"]["ecoinvent"]
scenario = config["lci"]["premise_scenario"]

ECOINVENT_DB = f"ecoinvent-{ei['version']}-{ei['system_model']}"
PREMISE_DB = (
    f"ei_{ei['system_model']}_{ei['version']}_"
    f"{scenario['model']}_{scenario['pathway']}_{scenario['year']}"
)
# The derived aluminium-conductor foreground, named as rules/lci.smk names it.
AL_CONDUCTOR_DB = f"{ECOINVENT_DB}-aluminium-conductor"

for name in [ECOINVENT_DB, PREMISE_DB, AL_CONDUCTOR_DB, ei["biosphere_name"]]:
    print(f"{name:55s} {len(bd.Database(name)):>8,d} activities")

ecoinvent-3.10-cutoff                                     23,523 activities
ei_cutoff_3.10_tiam-ucl_SSP2-RCP19_2040                   36,003 activities
ecoinvent-3.10-cutoff-aluminium-conductor                      5 activities
biosphere                                                  4,362 activities


## 2. Finding activities

`Database.search` is the fast free-text index (good for exploring). It is fuzzy, so it
is *not* how the workflow resolves activities — `run_lci.py` matches `name` and
`location` exactly and raises on 0 or >1 hits.

In [4]:
import pandas as pd

pd.set_option("display.max_columns", None)  # don't truncate columns
pd.set_option("display.max_colwidth", None)  # don't truncate cell contents
pd.set_option("display.width", None)  # don't wrap based on terminal width
pd.set_option("display.max_rows", 100)  # don't truncate rows

In [5]:
import pandas as pd


def search(term, database=ECOINVENT_DB, limit=99):
    """Free-text search, as a table of the fields you need to fill lci_info.csv."""
    hits = bd.Database(database).search(term, limit=limit)
    return pd.DataFrame(
        [
            {
                "name": a["name"],
                "location": a["location"],
                "reference product": a.get("reference product"),
                "unit": a["unit"],
            }
            for a in hits
        ]
    )


# The database matters: the electrolysers are premise's imported inventories, so this
# same call against ECOINVENT_DB finds nothing at all (section 8).
search("electrolyzer production", PREMISE_DB)

,name,location,reference product,unit
0,"electrolyzer production, 1MWe, SOEC, Stack",RER,"electrolyzer, 1MWe, SOEC, Stack",unit
1,"electrolyzer production, 1MWe, AEC, Stack",RER,"electrolyzer, 1MWe, AEC, Stack",unit
2,"electrolyzer production, 1MWe, PEM, Stack",RER,"electrolyzer, 1MWe, PEM, Stack",unit
3,"electrolyzer production, 1MWe, SOEC, Balance of Plant",RER,"electrolyzer, 1MWe, SOEC, Balance of Plant",unit
4,"electrolyzer production, 1MWe, AEC, Balance of Plant",RER,"electrolyzer, 1MWe, AEC, Balance of Plant",unit
5,"electrolyzer production, 1MWe, PEM, Balance of Plant",RER,"electrolyzer, 1MWe, PEM, Balance of Plant",unit
6,"treatment of electrolyzer stack, 1MWe, SOEC",RER,"used electrolyzer stack, 1MWe, SOEC",unit
7,"treatment of electrolyzer stack, 1MWe, PEM",RER,"used electrolyzer stack, 1MWe, PEM",unit
8,"treatment of electrolyzer stack, 1MWe, AEC",RER,"used electrolyzer stack, 1MWe, AEC",unit
9,"treatment of electrolyzer balance of plant, 1MWe, SOEC",RER,"used electrolyzer balance of plant, 1MWe, SOEC",unit


Exact lookup, the way the workflow does it. Use this to confirm a candidate row of
`data/lci_info.csv` will resolve before adding it.

In [6]:
def get_activity(name, location, database=ECOINVENT_DB):
    """The single activity with this exact name and location; raises otherwise."""
    matches = [
        a
        for a in bd.Database(database)
        if a["name"] == name and a["location"] == location
    ]
    if len(matches) != 1:
        raise ValueError(
            f"{len(matches)} matches in {database} for {name!r} @ {location!r}"
        )
    return matches[0]


activity = get_activity(
    "transmission network construction, electricity, high voltage", "CH"
)
dict(activity)

{'comment': 'This dataset describes the infrastructure (masts, cables etc.) of the electricity high voltage transmission network in Switzerland. It represents an average of underground cable and aerial lines per km of high voltage electricity transmission grid infrastructure. Switching stations are not included in this dataset, since switching stations for voltage transformation from high to medium voltage level are attributed to the medium voltage level and transformers from medium to low voltage level are attributed to the low voltage level.\nThe high voltage level is considred above 24 kV. Sometimes a highest voltage level is distinguished from the high voltage level grid; both are merged in this dataset. Typically large scale industry is supplied from this voltage level (Itten et al. 2014). Most power generation feeds into the high voltage grid.\nReference(s): René Itten, Rolf Frischknecht, Matthias Stucki, 2014, Life Cycle Inventories of Electricity Mixes and Grid\nGeography:  Dat

## 3. What is inside an activity

Exchanges are the direct (foreground) inputs only — one level down, not the full supply
chain. `technosphere` are products from other activities, `biosphere` are elementary
flows (resources in, emissions out), `production` is the activity's own output.

In [7]:
def exchanges(activity):
    """Direct exchanges of one activity.

    `minimum`/`maximum` and `preserve uncertainty` are empty on exchanges as
    ecoinvent/premise imported them, and populated on exchanges premise's `metals`
    update rewrote — see section 7.
    """
    return pd.DataFrame(
        [
            {
                "type": e["type"],
                "amount": e["amount"],
                "unit": e.input["unit"],
                "input": e.input["name"],
                "location": e.input.get("location"),
                "categories": e.input.get("categories"),
                "minimum": e.get("minimum"),
                "maximum": e.get("maximum"),
                "preserve uncertainty": e.get("preserve uncertainty"),
            }
            for e in activity.exchanges()
        ]
    ).sort_values(["type", "amount"], ascending=[True, False])


exchanges(activity).head(99)

,type,amount,unit,input,location,categories,minimum,maximum,preserve uncertainty
29,biosphere,1778.300,square meter-year,"Occupation, industrial area",None,"(natural resource, land)",None,None,None
32,biosphere,44.400,square meter,"Transformation, to industrial area",None,"(natural resource, land)",None,None,None
30,biosphere,24.700,square meter,"Transformation, from arable land, unspecified use",None,"(natural resource, land)",None,None,None
31,biosphere,19.700,square meter,"Transformation, from forest, unspecified",None,"(natural resource, land)",None,None,None
0,production,1.000,kilometer,"transmission network construction, electricity, high voltage",CH,None,None,None,None
21,technosphere,31000.000,kilogram,"market for steel, unalloyed",GLO,None,None,None,None
2,technosphere,11211.100,kilogram,"market for aluminium, wrought alloy",GLO,None,None,None,None
14,technosphere,1630.000,kilogram,market for lead,GLO,None,None,None,None
20,technosphere,1183.000,kilogram,"market for steel, low-alloyed",GLO,None,None,None,None
15,technosphere,1120.000,kilogram,market for light fuel oil,CH,None,None,None,None


Follow one input upstream — `e.input` is itself an activity, so you can walk the tree.

In [8]:
steel = next(
    e.input for e in activity.technosphere() if "steel" in e.input["name"]
)
print(steel)
exchanges(steel).head(15)

'building construction, hall, steel construction' (square meter, CH, None)


,type,amount,unit,input,location,categories,minimum,maximum,preserve uncertainty
0,production,1.00000,square meter,"building construction, hall, steel construction",CH,None,None,None,None
6,technosphere,238.00000,megajoule,"market for diesel, burned in building machine",GLO,None,None,None,None
3,technosphere,107.00000,kilogram,market for clay brick,RoW,None,None,None,None
11,technosphere,33.40000,kilogram,market for reinforcing steel,GLO,None,None,None,None
8,technosphere,33.00000,kilogram,market for fibre cement corrugated slab,GLO,None,None,None,None
2,technosphere,21.00000,kilogram,market for cement mortar,CH,None,None,None,None
13,technosphere,13.20000,kilogram,"market for stone wool, packed",GLO,None,None,None,None
7,technosphere,10.50000,kilowatt hour,"market for electricity, medium voltage",CH,None,None,None,None
10,technosphere,5.40000,kilogram,market for polystyrene foam slab,GLO,None,None,None,None
22,technosphere,5.33000,square meter,"zinc coating, coils",RER,None,None,None,None


## 4. The full inventory (what `run_lci.py` computes)

An LCA over a functional unit of 1 sums the *whole* upstream supply chain. The mineral
intensities come from the biosphere flows in the `natural resource / in ground`
category, which is what makes them extraction rather than emission.

In [9]:
from bw2calc import LCA


def resource_flows(activity, top=25):
    """Cumulative in-ground resource extraction for one unit of `activity`."""
    lca = LCA({activity: 1})
    lca.lci()

    reverse = {index: key for key, index in lca.biosphere_dict.items()}
    rows, _ = lca.inventory.nonzero()
    records = []
    for row in sorted(set(rows)):
        flow = bd.get_activity(reverse[row])
        records.append(
            {
                "flow name": flow["name"],
                "categories": " / ".join(flow["categories"]),
                "unit": flow["unit"],
                "amount": lca.inventory[row, :].sum(),
            }
        )
    flows = pd.DataFrame(records)
    flows = flows[flows["categories"] == RESOURCE_CATEGORY]
    return flows.sort_values("amount", ascending=False).head(top)


# Divide by lci_info.csv's `multiplier` to get the per-MW (functional unit) intensity.
resource_flows(activity)

,flow name,categories,unit,amount
1863,Gangue,natural resource / in ground,kilogram,369003.841772
208,Gravel,natural resource / in ground,kilogram,154009.724842
112,"Coal, hard",natural resource / in ground,kilogram,79654.919028
70,Calcite,natural resource / in ground,kilogram,35596.794812
1804,Iron,natural resource / in ground,kilogram,33847.086410
411,Shale,natural resource / in ground,kilogram,14344.981551
2171,"Oil, crude",natural resource / in ground,kilogram,10598.918620
2170,"Gas, natural",natural resource / in ground,standard cubic meter,9639.127164
1802,Aluminium,natural resource / in ground,kilogram,9211.958815
111,"Coal, brown",natural resource / in ground,kilogram,8595.535074


## 5. Checking `data/lci_info.csv` against the store

Every row of the info table must resolve to exactly one activity in its database.
This is the same check `run_lci` makes, run over the whole table at once.

In [10]:
# Mirrors DB_BY_SOURCE in rules/lci.smk — every source token in lci_info.csv.
db_by_source = {
    "ecoinvent": ECOINVENT_DB,
    "premise": PREMISE_DB,
    "al_conductor": AL_CONDUCTOR_DB,
}

lci_info = pd.read_csv(root / "data/lci_info.csv")

counts = []
for source, group in lci_info.groupby(LciInfo.database):
    db = list(bd.Database(db_by_source[source]))
    for _, row in group.iterrows():
        matches = [
            a
            for a in db
            if a["name"] == row[LciInfo.activity]
            and a["location"] == row[LciInfo.location]
        ]
        counts.append({**row, "matches": len(matches)})

checked = pd.DataFrame(counts)
checked[checked["matches"] != 1]

,activity,database,unit,multiplier,location,sub_technology,matches


## 6. ecoinvent vs. the premise database

The same activity name exists in both, with premise's rewired background. Comparing
them is how you tell a real foreground difference from a background artefact — in the
prospective database, trace flows of metals leak into technologies that do not
physically contain them (e.g. ~1e-05 kg/MW of iridium on an *alkaline* electrolyser).
Do not read those as bills of materials.

In [11]:
name, location = (
    "gas power plant construction, combined cycle, 400MW electrical",
    "RER",
)

comparison = pd.concat(
    {
        db: resource_flows(get_activity(name, location, db), top=None).set_index(
            "flow name"
        )["amount"]
        for db in [ECOINVENT_DB, PREMISE_DB]
    },
    axis=1,
)
comparison["ratio"] = comparison[PREMISE_DB] / comparison[ECOINVENT_DB]
comparison.sort_values(ECOINVENT_DB, ascending=False).head(20)

,ecoinvent-3.10-cutoff,ei_cutoff_3.10_tiam-ucl_SSP2-RCP19_2040,ratio
flow name,,,
Gangue,8.398951e+07,8.428058e+07,1.003466
Gravel,1.907702e+07,1.718631e+07,0.900891
"Coal, hard",1.702432e+07,9.261895e+06,0.544039
Iron,1.069106e+07,7.807363e+06,0.730270
"Oil, crude",7.566077e+06,7.035014e+06,0.929810
Calcite,6.465611e+06,5.827262e+06,0.901270
"Gas, natural",3.826039e+06,2.499313e+06,0.653238
Shale,3.127050e+06,1.793550e+06,0.573560
"Coal, brown",2.239464e+06,3.508849e+05,0.156683


## 7. Auditing a premise-modified activity

Section 6 covers background rewiring — trace metals leaking in through changed
electricity and hydrogen supply chains. This section is about a *different* premise
effect that produces the same symptom and needs a different check.

`scripts/build_premise.py:39` calls `ndb.update()` with no argument, which runs **every**
premise transformation, including `metals`. That one does not touch the background: it
**overwrites the bill of materials** of named datasets with literature metal
intensities. The result is a foreground exchange, so it survives every test that looks
for a background artefact, and it looks exactly like a real bill of materials — because
structurally it is one. The only way to tell is to go back to what premise imported.

Worked example: the alkaline electrolyser stack, whose 234.5 kg/MW of silver makes H₂
electrolysis 82% of this study's silver demand and 98% of its platinum. Of the 27 rows
in `data/lci_info.csv` this is the only one affected — the nuclear PWR is in premise's
overlay list too, but the study reads it from ecoinvent, and the batteries resolve
through `battery cell production`, which is overlaid under an `EV` row that carries no
silver.

In [12]:
AEC = get_activity(
    "electrolyzer production, 1MWe, AEC, Stack", "RER", PREMISE_DB
)
exchanges(AEC)

,type,amount,unit,input,location,categories,minimum,maximum,preserve uncertainty
0,production,1.000000,unit,"electrolyzer production, 1MWe, AEC, Stack",RER,None,NaN,NaN,None
5,technosphere,95553.312500,kilowatt hour,"market group for electricity, low voltage",WEU,None,NaN,NaN,None
2,technosphere,20194.400391,kilogram,"market for sheet rolling, chromium steel",GLO,None,NaN,NaN,None
3,technosphere,20194.400391,kilogram,"market for steel, chromium steel 18/8, hot rolled",GLO,None,NaN,NaN,None
9,technosphere,2458.250000,kilogram,"market for nickel, class 1",World,None,2000.000000,3166.666667,True
11,technosphere,602.693232,kilogram,market for potassium hydroxide,GLO,None,229.032125,607.585223,True
13,technosphere,366.663691,kilogram,market for zirconium oxide,GLO,None,NaN,NaN,None
12,technosphere,256.007758,kilogram,market for silver,World,None,37.015516,475.000000,True
1,technosphere,144.199997,kilogram,market for tetrafluoroethylene,GLO,None,NaN,NaN,None
6,technosphere,75.000000,kilogram,"market for aluminium, cast alloy",World,None,NaN,NaN,None


### Which exchanges premise wrote, and from what

`preserve uncertainty` in the table above is the fingerprint: exchanges as imported have
`uncertainty type: 0` and no such key, while everything the `metals` update touched
carries `loc`/`minimum`/`maximum` and `preserve uncertainty: True`. (Platinum is the
exception that proves it — its min equals its max, so premise wrote no distribution.)

Cross-referencing against `metals_db.csv` names all eight. `spread` is max/min: silver's
own stated range covers a factor of **13**, which is not a measurement of anything.

Two reading notes. Zirconium is the one row where `in dataset` and `premise median`
differ — premise converts elemental Zr to the ZrO₂ market via `conversion_factors.xlsx`
(271.449 × 123.22/91.224 = 366.66). And potassium's median exceeds its own max because
premise takes the median over the studies, then the KOH conversion is applied on top.

In [13]:
import premise

PREMISE_DATA = Path(premise.__file__).parent / "data"

# The AEL row is what premise applied to this dataset; conversion_factors.xlsx sets
# 1 unit == 1 MW for it, so these kg/MW figures land unscaled.
metals_db = pd.read_csv(PREMISE_DATA / "metals" / "metals_db.csv")
ael = metals_db[metals_db.origin_var == "AEL"].set_index("metal")

rewritten = []
for e in AEC.technosphere():
    match = [m for m in ael.index if m.lower() in e.input["name"].lower()]
    if match:
        metal = match[0]
        rewritten.append(
            {
                "metal": metal,
                "exchange": e.input["name"],
                "in dataset": e["amount"],
                "premise median": ael.loc[metal, "median"],
                "min": ael.loc[metal, "min"],
                "max": ael.loc[metal, "max"],
                "spread": ael.loc[metal, "max"] / ael.loc[metal, "min"],
            }
        )

pd.DataFrame(rewritten).sort_values("spread", ascending=False)

,metal,exchange,in dataset,premise median,min,max,spread
6,Silver,market for silver,256.007758,256.007758,37.015516,475.000000,12.832456
5,Potassium,market for potassium hydroxide,602.693232,420.000000,159.606061,423.409091,2.652838
1,Cobalt,market for cobalt,6.700000,6.700000,4.500000,8.900000,1.977778
3,Nickel,"market for nickel, class 1",2458.250000,2458.250000,2000.000000,3166.666667,1.583333
0,Aluminium,"market for aluminium, cast alloy",75.000000,75.000000,75.000000,75.000000,1.000000
2,Graphite,market for graphite,71.666667,71.666667,71.666667,71.666667,1.000000
4,Platinum,market for platinum,1.300000,1.300000,1.300000,1.300000,1.000000
7,Zirconium,market for zirconium oxide,366.663691,271.449324,271.449324,271.449324,1.000000


### What premise imported vs. what is in the store

This is the decisive comparison, and the reason the audit cannot be done from the store
alone. `lci-hydrogen-electrolysis.xlsx` is the inventory premise *imports* — Gerloff
(2021), the study the dataset's own `comment` field cites. The store holds that
inventory **after** `update()` has run over it.

A `NaN` on the left is an exchange the `metals` update invented: it appears in no
published inventory of this device. Silver and platinum are both in that column.
Gerloff's own answer to "what catalyses this stack" is the 2884.9 kg of nickel, and no
precious metal at all.

In [ ]:
INVENTORY = (
    PREMISE_DATA / "additional_inventories" / "lci-hydrogen-electrolysis.xlsx"
)
book = pd.read_excel(INVENTORY, header=None)
# Each dataset is a block starting at an "Activity" row.
BLOCK_STARTS = book.index[book[0].astype(str).eq("Activity")].tolist()


def imported_inventory(name):
    """The technosphere exchanges premise imports for `name`, before `update()` runs.

    Read from premise's own additional-inventories workbook, which is the only record
    of the dataset as published — the store holds it after the `metals` overlay.
    """
    start = next(i for i in BLOCK_STARTS if book.iat[i, 1] == name)
    end = next((i for i in BLOCK_STARTS if i > start), len(book))
    block = book.loc[start : end - 1]

    header = block.index[block[0].astype(str).eq("name")][0]
    rows = block.loc[header + 1 :, [0, 1, 5]]
    rows.columns = ["input", "amount", "type"]
    rows = rows[rows["type"] == "technosphere"].dropna(subset=["input"])
    return rows.groupby("input")["amount"].sum()


def stored_inventory(activity):
    """The same dataset's technosphere exchanges as the store holds them."""
    return pd.DataFrame(
        [(e.input["name"], e["amount"]) for e in activity.technosphere()],
        columns=["input", "amount"],
    ).groupby("input")["amount"].sum()


pd.DataFrame(
    {
        "imported by premise": imported_inventory(AEC["name"]),
        "in the store": stored_inventory(AEC),
    }
).sort_values("in the store", ascending=False)

### Where the numbers came from

`metals_db.csv` is only the digest. `SI_2_Material_requirements.xlsx` keeps one column
per underlying study, so each value is traceable, and the `References` sheet resolves
the source keys.

Read the silver row against the reference titles. Its two values come from
`Staffel_2010` and `Stahl_2016` — both titled *"Life cycle assessment of an alkaline
**fuel cell** CHP system"*, and both carrying the same DOI, so one of the two citations
is a copy-paste and the real count is n≈1. Silver is a standard alkaline fuel cell
oxygen-reduction cathode catalyst; an alkaline **electrolyser** uses nickel electrodes.

`Koj_2017` is the one source here that is actually about alkaline water electrolysis. It
supplies the aluminium, graphite, nickel and zirconium — and no silver and no platinum.

In [15]:
SI = PREMISE_DATA / "metals" / "SI_2_Material_requirements.xlsx"

# Cols 0-7 are the summary statistics; everything from col 8 on is one source each.
sheet = pd.read_excel(SI, sheet_name="AEL", header=None)
sources = sheet.iloc[0, 8:].tolist()
per_source = sheet.iloc[3:, [0, 3, 4, 5, 6, 7] + list(range(8, sheet.shape[1]))]
per_source.columns = ["material", "mean", "median", "min", "max", "count"] + sources
display(per_source)

references = pd.read_excel(SI, sheet_name="References", header=None, usecols=[1, 2, 3])
references.columns = ["key", "title", "link"]
references[references.key.isin(sources)]

,material,mean,median,min,max,count,Grandell_2016,Staffel_2010,Stahl_2016,Wuppertal_2014,Koj_2017
3,Aluminium,75,75,75,75,1,NaN,NaN,NaN,NaN,75
4,Cobalt,6.7,6.7,4.5,8.9,2,8.9,NaN,4.5,NaN,NaN
5,Graphite,71.666667,71.666667,71.666667,71.666667,1,NaN,NaN,NaN,NaN,71.666667
6,Nickel,2520.791667,2458.25,2000,3166.666667,4,NaN,2007.5,2909,2000,3166.666667
7,Platinum,1.3,1.3,1.3,1.3,1,NaN,1.3,NaN,NaN,NaN
8,Potassium,334.338384,420,159.606061,423.409091,3,NaN,423.409091,159.606061,420,NaN
9,Silver,256.007758,256.007758,37.015516,475,2,NaN,475,37.015516,NaN,NaN
10,Zirconium,271.449324,271.449324,271.449324,271.449324,1,NaN,NaN,NaN,NaN,271.449324


,key,title,link
20,Grandell_2016,Role of critical metals in the future markets of clean energy technologies,https://www.sciencedirect.com/science/article/abs/pii/S0960148116302816?via%3Dihub
32,Koj_2017,Site-Dependent Environmental Impacts of Industrial Hydrogen Production by Alkaline Water Electrolysis,https://doi.org/10.3390/en10070860
58,Staffel_2010,Life cycle assessment of an alkaline fuel cell CHP system,https://doi.org/10.1016/j.ijhydene.2009.12.135
59,Stahl_2016,Life cycle assessment of an alkaline fuel cell CHP system,https://doi.org/10.1016/j.ijhydene.2009.12.135
74,Wuppertal_2014,KRESSE - Kritische mineralische Ressourcen und Stoffströme bei der Transformation des deutschen Energieversorgungssystems,https://epub.wupperinst.org/frontdoor/deliver/index/docId/5419/file/5419_KRESSE.pdf


### The cross-check that does not depend on reading titles

premise gives the **alkaline** stack 1.3 kg/MW platinum against 0.30 kg/MW for its own
**PEM** stack. PEM is the PGM technology; being PGM-free is alkaline's whole selling
point, so that ordering is inverted. The same row hands alkaline 256 kg/MW of silver
while the PEM sheet has none.

Note also that `PEMEL` carries a 2050 row and `AEL` does not, and that premise does not
interpolate between them. `metals.py` builds its factors with
`self.metals.interp(year=self.year, method="nearest")` over year coordinates of 2020 and
2050, so the loadings are a **step**: a database up to 2035 gets the 2020 figures, and
2036 onwards gets 2050. This 2040 database therefore holds PEM's 2050 values exactly
(Ir 0.195, Pt 0.0238, Ti 37.0 — see section 8), while AEL, having no 2050 row, is
forward-filled from 2020 and gets no thrifting at all.

In [16]:
metals_db[
    metals_db.origin_var.isin(["AEL", "PEMEL"])
    & metals_db.metal.isin(["Silver", "Platinum", "Iridium", "Nickel"])
].sort_values(["metal", "origin_var", "year"])

,metal,unit,year,mean,median,min,max,origin_var
215,Iridium,kg/MW,2020.0,1.223899,0.738017,0.339988,4.166667,PEMEL
220,Iridium,kg/MW,2050.0,0.194994,0.194994,0.050000,0.339988,PEMEL
226,Nickel,kg/MW,2020.0,2520.791667,2458.250000,2000.000000,3166.666667,AEL
227,Platinum,kg/MW,2020.0,1.300000,1.300000,1.300000,1.300000,AEL
216,Platinum,kg/MW,2020.0,0.373189,0.304688,0.037500,1.041667,PEMEL
221,Platinum,kg/MW,2050.0,0.023750,0.023750,0.010000,0.037500,PEMEL
229,Silver,kg/MW,2020.0,256.007758,256.007758,37.015516,475.000000,AEL


## 8. Electrolyser types: what the store has

**ecoinvent 3.10 holds no electrolyser-production dataset at all**.
The devices arrive with premise, which imports `lci-hydrogen-electrolysis.xlsx` (Gerloff 2021) as an additional inventory, so they exist only in `PREMISE_DB`.

In [17]:
def names_matching(term, database):
    """Every distinct (name, location, unit) in `database` whose name contains `term`.

    `search` reads a text index built per database; this walks the database itself, so
    it is what answers "is it in here at all" when a search comes back empty.
    """
    return pd.DataFrame(
        sorted(
            {
                (a["name"], a["location"], a["unit"])
                for a in bd.Database(database)
                if term.lower() in a["name"].lower()
            }
        ),
        columns=["name", "location", "unit"],
    )


for database in (ECOINVENT_DB, PREMISE_DB):
    print(
        f"{database:45s} {len(names_matching('electrolyzer', database)):3d} activities "
        "with 'electrolyzer' in the name"
    )

names_matching("electrolyzer production", PREMISE_DB)

ecoinvent-3.10-cutoff                           0 activities with 'electrolyzer' in the name
ei_cutoff_3.10_tiam-ucl_SSP2-RCP19_2040        12 activities with 'electrolyzer' in the name


,name,location,unit
0,"electrolyzer production, 1MWe, AEC, Balance of Plant",RER,unit
1,"electrolyzer production, 1MWe, AEC, Stack",RER,unit
2,"electrolyzer production, 1MWe, PEM, Balance of Plant",RER,unit
3,"electrolyzer production, 1MWe, PEM, Stack",RER,unit
4,"electrolyzer production, 1MWe, SOEC, Balance of Plant",RER,unit
5,"electrolyzer production, 1MWe, SOEC, Stack",RER,unit


Six datasets: three technologies — **AEC** (alkaline), **PEM** and **SOEC** (solid
oxide) — each split into **Stack** and **Balance of Plant**. All `RER`, all in `unit`,
and premise's `conversion_factors.xlsx` sets 1 unit == 1 MW, which is why
`data/lci_info.csv` carries the AEC stack against `MW` with a `multiplier` of 1.0.

The study reads the **stack alone** (`H2 Electrolyzer, AEC`), so a type scenario is
stack-for-stack: same boundary, one row per type. The balance of plant is a separate
decision — it is not in the study for AEC either, and adding it for one type only would
confound the material swap with a change of boundary. It is in the table below so the
size of what is being left out is visible.

The kg/MW below are the flows the workflow would actually charge: in-ground flows on
routes `data/lci_route_info.csv` declares `embodied`, which is what
`select_embodied_flows` keeps.

In [18]:
from info_schemas import (
    FlowRoute,
    LciResourceFlow,
    MineralGroup,
    MineralInfo,
    RouteInfo,
)
from run_lci import mineral_flows_by_route

mineral_info = pd.read_csv(root / "data/mineral_info.csv")
STUDY_MINERALS = set(
    mineral_info.loc[
        mineral_info[MineralInfo.group] != MineralGroup.helper, MineralInfo.mineral
    ]
)

ELECTROLYSERS = [
    f"electrolyzer production, 1MWe, {kind}, {part}"
    for kind in ("AEC", "PEM", "SOEC")
    for part in ("Stack", "Balance of Plant")
]


def flows_by_route(name, database=PREMISE_DB, location="RER"):
    """One unit's study-mineral in-ground flow, per route.

    Calls run_lci's own function, so these are the workflow's numbers rather than a
    second implementation of them.
    """
    lca = LCA({get_activity(name, location, database): 1})
    lca.lci()
    flows = pd.DataFrame(mineral_flows_by_route(lca, bd, STUDY_MINERALS))
    return flows.groupby([FlowRoute.mineral, FlowRoute.route], as_index=False)[
        FlowRoute.amount
    ].sum()


routes = pd.concat(
    {name: flows_by_route(name) for name in ELECTROLYSERS}, names=["dataset", None]
).reset_index("dataset")
routes["dataset"] = routes["dataset"].str.removeprefix(
    "electrolyzer production, 1MWe, "
)

# Only the routes the study calls embodied become demand, so that is what to compare.
verdicts = pd.read_csv(root / "data/lci_route_info.csv")
charged = routes.merge(verdicts, on=[RouteInfo.mineral, RouteInfo.route], how="left")

embodied = (
    charged[charged[RouteInfo.verdict] == LciResourceFlow.embodied]
    .pivot_table(
        index=FlowRoute.mineral,
        columns="dataset",
        values=FlowRoute.amount,
        aggfunc="sum",
    )
    .reindex(
        columns=[
            name.removeprefix("electrolyzer production, 1MWe, ")
            for name in ELECTROLYSERS
        ]
    )
    .fillna(0.0)
)
embodied.sort_values("AEC, Stack", ascending=False).round(3)

dataset,"AEC, Stack","AEC, Balance of Plant","PEM, Stack","PEM, Balance of Plant","SOEC, Stack","SOEC, Balance of Plant"
mineral,,,,,,
Nickel,5833.227,1051.816,28.059,674.765,1689.655,2438.495
Aluminium,802.346,307.370,265.858,332.756,207.180,698.122
Copper,268.232,569.982,72.369,339.026,90.932,459.176
Silver,234.536,0.095,0.050,0.080,0.069,0.161
Manganese,47.784,181.637,10.710,105.112,27.942,252.475
Cobalt,7.079,0.203,0.084,0.138,5.979,0.480
Tin,4.492,2.486,1.207,2.082,1.634,3.953
Platinum,1.243,0.000,0.023,0.000,0.000,0.000
Vanadium,0.598,0.166,0.193,0.152,0.270,0.337


### Reading the table

Nickel is what separates the three stacks: 5,833 kg/MW on AEC against 1,690 on SOEC and
**28** on PEM. Against that, PEM is the only one carrying iridium (0.195 kg/MW), and
SOEC the only one carrying neodymium and praseodymium at any scale (13.3 and 12.2
kg/MW).

None of those three bills is the inventory premise imported: the `metals` update
rewrites all three stacks and leaves all three balance-of-plant datasets alone. Section
9 sets the three against Gerloff (2021), whose figures premise imports. Before reading
any of this as a scenario:

1. **The AEC silver and platinum are premise's, not Gerloff's** (sections 7 and 9). The
   234.5 kg/MW of silver is 82% of this study's silver demand. A switch to PEM would
   appear to delete nearly all of it — an artefact of the overlay, not a material
   saving, and it would be the scenario's headline number.
2. **The vintages do not match.** `metals_db.csv` gives `PEMEL` a 2050 row and `AEL`
   only a 2020 one, and premise steps to the nearest of the two rather than
   interpolating, so this 2040 database holds PEM's 2050 values (Ir 0.195, Pt 0.0238,
   Ti 37.0, against 2020 medians of 0.738, 0.305, 414) against an unthrifted 2020
   alkaline.
3. **SOEC's neodymium is a passenger**, not a magnet — next cell.

In [19]:
soec = get_activity("electrolyzer production, 1MWe, SOEC, Stack", "RER", PREMISE_DB)

# Which purchased oxide brings the Nd and Pr in, and whether premise wrote the amount.
REE_OXIDES = ("cerium", "lanthanum", "praseodymium", "yttrium", "samarium")

bought = []
for exchange in soec.technosphere():
    if not any(oxide in exchange.input["name"] for oxide in REE_OXIDES):
        continue
    lca = LCA({exchange.input: exchange["amount"]})
    lca.lci()
    flows = pd.DataFrame(mineral_flows_by_route(lca, bd, STUDY_MINERALS))
    bought.append(
        {
            "input": exchange.input["name"],
            "kg": exchange["amount"],
            "premise-written": bool(exchange.get("preserve uncertainty")),
            **flows.groupby(FlowRoute.mineral)[FlowRoute.amount]
            .sum()
            .reindex(["Neodymium", "Praseodymium"])
            .fillna(0.0)
            .round(3)
            .to_dict(),
        }
    )

pd.DataFrame(bought).sort_values("Neodymium", ascending=False)

,input,kg,premise-written,Neodymium,Praseodymium
1,market for samarium-europium-gadolinium oxide,22.900000,False,8.075,2.706
0,market for samarium-europium-gadolinium oxide,14.800000,False,5.219,1.749
2,market for cerium oxide,225.086111,False,0.002,0.000
5,market for praseodymium oxide,9.000000,False,0.001,7.690
3,market for lanthanum oxide,13.121857,True,0.000,0.000
4,market for yttrium oxide,7.640760,True,0.000,0.000


### What the SOEC rare earths actually are

37.7 kg of `samarium-europium-gadolinium oxide` carries 13.3 kg of neodymium and 4.5 kg
of praseodymium out of the same ore. Neither is premise's doing — `premise-written` is
`False`, this is the imported inventory going through ecoinvent's REE chain — and the
praseodymium the stack really buys is the 9 kg of praseodymium oxide, worth 7.7 kg.

That matters because the route is `rare earth oxide concentrate, 50% REO`, which
`data/lci_route_info.csv` declares **embodied** for neodymium, correctly: for a magnet
that route *is* the magnet's. Charged to SOEC unchanged, the study would book 13.3 kg/MW
of magnet-grade neodymium the device does not contain. It is the case where one verdict
per (mineral, route) does not separate the two readings, and it has to be settled before
SOEC can be a sub-technology.

Two further routes turn up undeclared above `lci.route_declaration_threshold`, so
`select_embodied_flows` would raise on them: `citric acid` on the SOEC stack (Ga, Mn,
Al, Tb, Dy, graphite) and `zeolite, powder` on the PEM balance of plant. The AEC stack,
already in the workflow, adds none.

In [20]:
threshold = config["lci"]["route_declaration_threshold"]

# Signed share of the activity's total for that mineral, as check_declared measures it.
totals = charged.groupby(["dataset", FlowRoute.mineral])[FlowRoute.amount].transform(
    "sum"
)
charged["share"] = charged[FlowRoute.amount] / totals

missing = charged[charged[RouteInfo.verdict].isna() & (charged["share"] >= threshold)]
missing.sort_values("share", ascending=False)[
    ["dataset", FlowRoute.mineral, FlowRoute.route, FlowRoute.amount, "share"]
]

,dataset,mineral,route,amount,share
2610,"PEM, Balance of Plant",Praseodymium,"zeolite, powder",2.080434e-04,0.030608
2478,"PEM, Balance of Plant",Neodymium,"zeolite, powder",6.383722e-04,0.021004
2455,"PEM, Balance of Plant",Molybdenum,"zeolite, powder",1.059301e-02,0.014605
2164,"PEM, Balance of Plant",Cadmium,"zeolite, powder",1.355689e-05,0.009018
2633,"PEM, Balance of Plant",Selenium,"zeolite, powder",9.728180e-04,0.006836
2721,"PEM, Balance of Plant",Terbium,"zeolite, powder",3.372543e-06,0.006732
2985,"SOEC, Stack",Gallium,citric acid,1.487912e-04,0.005568
2286,"PEM, Balance of Plant",Dysprosium,"zeolite, powder",5.713127e-06,0.004726
2674,"PEM, Balance of Plant",Silver,"zeolite, powder",3.095607e-04,0.003795
2587,"PEM, Balance of Plant",Platinum,"zeolite, powder",8.231531e-07,0.003595


### Wiring a type scenario

What a PEM (or SOEC) sub-technology needs, all of it together or the DAG build fails:

- `data/lci_info.csv` — one row per stack: `premise`, `MW`, multiplier `1.0`, `RER`.
- `config/default.yaml` `sub_technology_info` — `"H2 Electrolyzer, PEM": lci`.
- `config/default.yaml` `technology_info.h2_electrolysis.sub_technologies` — add it.
- `data/market_share_scenarios.csv` — a row for the new sub-technology in **every** column
  (0 where it is not used; today `H2 Electrolyzer, AEC` is 1 in all nine), plus the new
  scenario column setting PEM → 1 and AEC → 0.
- `config/default.yaml` `scenarios.<name>` — `market_share_scenario` pointing at that
  column, a `group`, and `scenario_type: substitution`; `check_scenario_types` derives the
  type from how the scenario is built and rejects a mismatch.
- `data/lci_route_info.csv` — the routes above.
- `data/source/sources.md` — the scenario-columns table, alongside the other scenarios.


## 9. premise against the study it imports

Section 7 audited one dataset against its source. This does all six, and there is a
third column the store cannot supply: what Gerloff (2021) actually published. The
inventory premise imports reproduces his figures faithfully (one exception, below), so
the store-versus-workbook diff below *is* the overlay's edit, and the workbook is the
published bill.

`update()` rewrites all three **stacks** and leaves all three **balance-of-plant**
datasets untouched, and a scan of the whole database finds none of these amounts
anywhere else, so the edit is confined to the six datasets below.

In [ ]:
changed = []
for name in ELECTROLYSERS:
    activity = get_activity(name, "RER", PREMISE_DB)
    both = pd.concat(
        {
            "imported": imported_inventory(name),
            "store": stored_inventory(activity),
        },
        axis=1,
    )
    both["ratio"] = both["store"] / both["imported"]
    both = both[~((both["ratio"] - 1).abs() < 1e-6)]  # unchanged rows are not news
    changed.append(both.assign(dataset=name.removeprefix("electrolyzer production, 1MWe, ")))

pd.concat(changed).set_index("dataset", append=True).swaplevel().round(4)

### What the differences are

Against Gerloff's own figures (`scratch/gerloff_2021_SI.docx`, Figs. 1–3, whose values
premise's workbook reproduces), the `metals` update does four distinguishable things.

**It invents rows.** The AEC stack gains silver 256.0, KOH 602.7, aluminium 75.0,
graphite 71.7, cobalt 6.7 and platinum 1.3 kg/MW; none is in the published inventory.
Silver and platinum come from `Staffel_2010` and `Stahl_2016`, which are the *same
paper* on an alkaline **fuel cell** (identical title and DOI, so premise's n = 2 is
really n = 1). The PEM stack gains 200.0 kg/MW of potassium hydroxide, whose only source
in the `PEMEL` sheet is that same alkaline fuel cell paper — a PEM cell has a solid acid
membrane and no liquid alkali at all.

**It double-counts one material.** Gerloff states that yttrium had no ecoinvent dataset
and was *replaced* by 37.7 kg of samarium-europium-gadolinium concentrate. premise adds
`market for yttrium oxide` 7.64 kg and leaves the proxy in place, so the SOEC stack now
buys its yttria twice — and the proxy is what carries the 13.3 kg/MW of neodymium.

**It overwrites device-specific values with cross-study medians**, including where the
study's own source is one of the studies averaged. The AEC diaphragm's 73 kg of
zirconium oxide becomes 366.7 (×5.0, from Koj 2017's generic figure); SOEC's zirconium
oxide 170.7 → 366.6, nickel 144.1 → 314.9, cerium oxide 91.5 → 225.1, lanthanum oxide
21 → 13.1, strontium carbonate 21 → 4.0; AEC nickel 2884.9 → 2458.3.

**It thrifts by a step, not a trajectory** (section 7), so PEM and SOEC arrive here on
2050 loadings and AEC on 2020 ones.

One import error sits upstream of all of that: Gerloff's PEM cathode carries **0.075**
kg of platinum and premise's workbook says **0.75**. It does not reach this database —
the overlay overwrites the row with 0.0238 — but it is wrong in any database built
without the `metals` update.

Only the AEC stack is in `data/lci_info.csv`, so only its differences move this study's
results. The next cell prices them.

In [ ]:
# What the AEC stack's intensity would be on the published bill of materials, by
# reversing each overwritten row. An LCI is linear in its exchanges, so the effect of a
# row is (amount premise wrote - amount premise imported) x that input's own intensity.
OVERLAY_DELTAS = {
    "market for nickel, class 1": 2458.25 - 2884.9,
    "market for zirconium oxide": 366.663691 - 73.0,
    "market for aluminium, cast alloy": 75.0,
    "market for cobalt": 6.7,
    "market for graphite": 71.666667,
    "market for platinum": 1.3,
    "market for potassium hydroxide": 602.693232,
    "market for silver": 256.007758,
}


def embodied(demand):
    """Study-mineral intensity of a demand vector, embodied routes only."""
    lca = LCA(demand)
    lca.lci()
    flows = pd.DataFrame(mineral_flows_by_route(lca, bd, STUDY_MINERALS))
    verdict = flows.merge(
        verdicts, on=[RouteInfo.mineral, RouteInfo.route], how="left"
    )
    return (
        verdict[verdict[RouteInfo.verdict] == LciResourceFlow.embodied]
        .groupby(FlowRoute.mineral)[FlowRoute.amount]
        .sum()
    )


contribution = pd.DataFrame(
    {
        exchange.input["name"]: embodied({exchange.input: OVERLAY_DELTAS[exchange.input["name"]]})
        for exchange in AEC.technosphere()
        if exchange.input["name"] in OVERLAY_DELTAS
    }
).fillna(0.0)

as_is = embodied({AEC: 1})
audit = pd.DataFrame(
    {
        "premise (store)": as_is,
        "on Gerloff's bill": as_is - contribution.sum(axis=1).reindex(as_is.index).fillna(0.0),
    }
)
audit["ratio"] = audit["premise (store)"] / audit["on Gerloff's bill"]
audit.sort_values("premise (store)", ascending=False).round(3)